# F07: Healthcare Revenue Cycle Management

## What You'll Learn

Revenue Cycle Management (RCM) is the financial backbone of healthcare — it tracks
every dollar from patient encounter through claim submission to final payment. In this
notebook you will:

1. **Generate** a healthcare dataset at medium scale
2. **Explore** the patient, encounter, claim, and claim_line tables
3. **Simulate** a claim processing pipeline with status transitions
4. **Export** SQL INSERT statements for data warehouse loading
5. **Build** a star schema with `dim_patient`, `dim_provider`, and `fact_claim`
6. **Discuss** why HIPAA-safe synthetic data matters for RCM analytics

## Tables We'll Use

| Table | RCM Role |
|---|---|
| `patient` | Demographics, insurance information |
| `encounter` | Office visits, ED, inpatient stays |
| `claim` | Payer claims with status and charge amounts |
| `claim_line` | Line-item detail (CPT codes, units, charges) |
| `provider` | Rendering and billing physicians |
| `facility` | Service locations |

## Prerequisites

- `sqllocks-spindle` installed
- Familiarity with healthcare billing concepts (helpful but not required)

## Time Estimate

**~20 minutes**

In [ ]:
# Uncomment the line below if sqllocks-spindle is not yet installed.
# %pip install sqllocks-spindle

print("Installation cell ready. Uncomment the pip line above if needed.")

## Step 1 — Generate Healthcare Data at Medium Scale

**What we're about to do:** Generate the full healthcare domain at `medium` scale,
which produces enough data for meaningful RCM analytics — thousands of encounters
and claims.

**Why this matters:** RCM dashboards need volume to show realistic denial rates,
A/R aging distributions, and payer mix patterns. The `medium` scale hits the sweet
spot between speed and analytical richness.

In [ ]:
from sqllocks_spindle import Spindle, HealthcareDomain

spindle = Spindle()
data = spindle.generate(domain=HealthcareDomain(), scale="medium", seed=42)

print("Healthcare Domain — Medium Scale")
print("=" * 45)
for table_name, df in data.items():
    print(f"  {table_name:<15} {len(df):>8,} rows")
print(f"\nTotal tables: {len(data)}")
print(f"Total rows:   {sum(len(df) for df in data.values()):,}")

## Step 2 — Explore the Core RCM Tables

**What we're about to do:** Examine the patient, encounter, claim, and claim_line
tables — the four tables that form the backbone of any RCM data model.

**Why this matters:** Understanding the shape and relationships of these tables is
essential before building dashboards or ETL pipelines. Each table connects to the
next via foreign keys: patient -> encounter -> claim -> claim_line.

In [ ]:
import pandas as pd

# Patient demographics snapshot
patient = data["patient"]
print("=== Patient Table ===")
print(f"Columns: {list(patient.columns)}")
print(f"Rows: {len(patient):,}\n")
print(patient.head(3).to_string(index=False))

# Encounter types
encounter = data["encounter"]
print("\n\n=== Encounter Type Mix ===")
enc_mix = encounter["encounter_type"].value_counts()
enc_pct = encounter["encounter_type"].value_counts(normalize=True).mul(100).round(1)
for val in enc_mix.index:
    print(f"  {val:<15} {enc_mix[val]:>6,}  ({enc_pct[val]}%)")

# Claim status distribution
claim = data["claim"]
print("\n=== Claim Status Distribution ===")
status = claim["status"].value_counts()
status_pct = claim["status"].value_counts(normalize=True).mul(100).round(1)
for val in status.index:
    print(f"  {val:<12} {status[val]:>6,} claims  ({status_pct[val]}%)")

# Claim line detail
claim_line = data["claim_line"]
print(f"\n=== Claim Lines ===")
print(f"Total claim lines: {len(claim_line):,}")
print(f"Avg lines per claim: {len(claim_line) / len(claim):.1f}")
print(claim_line.head(3).to_string(index=False))

## Step 3 — Claim Processing Pipeline Simulation

**What we're about to do:** Join claims to encounters and patients to simulate an
RCM pipeline view — showing the journey from patient visit through claim adjudication.
We'll compute key RCM metrics: denial rate, average charge, and A/R by status.

**Why this matters:** These are the exact KPIs that RCM directors track daily.
Having realistic synthetic data lets you build and test dashboards without waiting
for production data access or navigating HIPAA compliance reviews.

In [ ]:
# Build the RCM pipeline view: patient -> encounter -> claim
pipeline = (
    claim
    .merge(encounter[["encounter_id", "patient_id", "encounter_type"]], on="encounter_id", how="left")
    .merge(patient[["patient_id", "gender", "insurance_type"]], on="patient_id", how="left")
)

# Key RCM metrics
total_charges = pipeline["total_charge"].sum()
denied = pipeline[pipeline["status"] == "denied"]
denial_rate = len(denied) / len(pipeline) * 100

print("RCM Pipeline Summary")
print("=" * 50)
print(f"Total claims:     {len(pipeline):>10,}")
print(f"Total charges:    ${total_charges:>12,.2f}")
print(f"Denial rate:      {denial_rate:>10.1f}%")
print(f"Avg charge/claim: ${pipeline['total_charge'].mean():>12,.2f}")

# A/R breakdown by status
print("\n=== Accounts Receivable by Status ===")
ar = pipeline.groupby("status")["total_charge"].agg(["count", "sum", "mean"])
ar.columns = ["Claims", "Total Charges", "Avg Charge"]
ar = ar.sort_values("Total Charges", ascending=False)
print(ar.to_string())

# Denial rate by encounter type
print("\n=== Denial Rate by Encounter Type ===")
for enc_type in pipeline["encounter_type"].unique():
    subset = pipeline[pipeline["encounter_type"] == enc_type]
    enc_denials = subset[subset["status"] == "denied"]
    rate = len(enc_denials) / len(subset) * 100 if len(subset) > 0 else 0
    print(f"  {enc_type:<15} {rate:.1f}%  ({len(enc_denials):,} / {len(subset):,})")

## Step 4 — Export SQL INSERT Statements for Warehouse Loading

**What we're about to do:** Use `to_sql_inserts()` to generate T-SQL INSERT statements
for loading the healthcare data into a SQL Server, Azure SQL, or Fabric Warehouse.

**Why this matters:** Many RCM systems use SQL databases as their analytical warehouse.
`to_sql_inserts()` generates DDL (CREATE TABLE) plus batched INSERT statements that
you can run directly against any T-SQL compatible endpoint.

In [ ]:
from sqllocks_spindle.output import PandasWriter
import os

writer = PandasWriter()
output_dir = "./spindle_healthcare_sql"

sql_files = writer.to_sql_inserts(
    tables=dict(data),
    output_dir=output_dir,
    schema_name="rcm",
    batch_size=500,
    include_ddl=True,
    sql_dialect="tsql",
)

print(f"Generated {len(sql_files)} SQL files in {output_dir}/\n")
for f in sql_files:
    size = os.path.getsize(f)
    print(f"  {os.path.basename(f):<35} {size:>10,} bytes")

# Preview the first few lines of one file
print(f"\n=== Preview: {os.path.basename(sql_files[0])} (first 15 lines) ===")
with open(sql_files[0], "r") as fh:
    for i, line in enumerate(fh):
        if i >= 15:
            print("  ...")
            break
        print(f"  {line.rstrip()}")

## Step 5 — Healthcare Star Schema

**What we're about to do:** Transform the 3NF healthcare data into a star schema
with `dim_patient`, `dim_provider`, `dim_facility`, and `fact_claim` (plus the
auto-generated `dim_date`).

**Why this matters:** Star schemas are the foundation of healthcare BI. Power BI,
Fabric DirectLake, and SSAS Tabular all expect dimension/fact models. This transform
gives you a warehouse-ready model from synthetic data in seconds.

In [ ]:
from sqllocks_spindle.transform import StarSchemaTransform

transform = StarSchemaTransform()
star = transform.transform(dict(data), HealthcareDomain().star_schema_map())

print(star.summary())

# Show dimensions
print("\n=== Dimension Tables ===")
for name, df in star.dimensions.items():
    print(f"  {name:<25} {len(df):>8,} rows  |  Columns: {len(df.columns)}")

# Show facts
print("\n=== Fact Tables ===")
for name, df in star.facts.items():
    print(f"  {name:<25} {len(df):>8,} rows  |  Columns: {len(df.columns)}")

# Date dimension
print(f"\n=== dim_date ===")
print(f"  Date range: {star.date_dim['date'].min()} to {star.date_dim['date'].max()}")
print(f"  Total days: {len(star.date_dim):,}")

## Why HIPAA-Safe Synthetic Data Matters

Working with real Protected Health Information (PHI) for analytics development is
a compliance minefield:

- **Access controls**: PHI requires BAAs, role-based access, audit logging
- **De-identification**: Safe Harbor or Expert Determination methods are expensive
  and error-prone
- **Environment restrictions**: PHI often can't leave production networks, making
  dev/test painful
- **Breach risk**: Every copy of real data is a potential breach vector

**Spindle's healthcare domain solves this** by generating data that:

- Contains **zero real PHI** — every patient, provider, and facility is synthetic
- Follows **realistic clinical distributions** (Census demographics, CMS diagnosis
  frequencies, industry-standard payer mixes)
- Maintains **full referential integrity** — JOINs work exactly as they would on
  production data
- Is **reproducible** — same seed produces identical datasets, enabling regression
  testing

You can share Spindle-generated healthcare data freely across teams, environments,
and cloud regions with no HIPAA compliance burden.

## What's Next?

You've built a complete RCM analytics pipeline from synthetic healthcare data —
from raw encounters through claim processing to a warehouse-ready star schema.

| Notebook | What You'll Learn |
|----------|-------------------|
| **T04: Healthcare Deep Dive** | Detailed exploration of every healthcare table and its distributions |
| **T06: Star Schema Export** | Deep dive into star schema mechanics, Parquet export |
| **T08: Fabric Quickstart** | Load this data directly into a Fabric Lakehouse |

**Key takeaways:**
- Spindle's healthcare domain generates realistic RCM data at any scale
- Denial rates, payer mixes, and encounter distributions mirror industry benchmarks
- `to_sql_inserts()` gives you T-SQL ready for any SQL Server/Fabric Warehouse
- `StarSchemaTransform` produces a BI-ready model with dim/fact tables
- Synthetic data eliminates HIPAA compliance barriers for dev/test workloads